# Data Description
- Author: 李芸榛（JenniferjjLee）
- Date first created: 20261108
- Background: 研究範圍內 108 站（`stations_in_field.csv` 的 107 站，加上臺大人文館，StationID 暫代 `-2`）每小時的 YouBike 借出量、還入量與淨流量，由 v6 台大周邊交易資料（2024、2025）彙總而成，並依 `trip_group` 拆分旅次類型。範圍外站點只收到和台大有往來的旅次，流量不完整，所以不列入。
- Unit of observation: 一個站點的一個小時（`station_id` × `datetime`，唯一且完整）。
- Sample period: 2024-01-01 00 時～2025-12-31 23 時，共 17,544 小時；108 站 × 17,544 小時 = 1,894,752 列，沒有借還的小時填 0。
- Known issues:
  - **時間只精確到小時**：原始交易的借還時間都是整點，所以只能做到小時層級。
  - **淨流量不含調度**：只反映騎乘，看不到調度車補車或載走，不能直接當作站上車數的變化。
  - **`in_service = 0` 不是沒人借**：表示該站當時還沒啟用（60,048 列）。啟用日 = 2024 年站位快照的 `first_seen` 和該站第一筆交易日期取較早者；臺大人文館從 2025-02-04 起。`捷運科技大樓站(台北教育大學)`、`臺大共同教室北側` 從 2024-01-02 起標為啟用，但分別到 2024-07、2024-06 才有第一筆交易，之前雖然啟用、借還幾乎都是 0。
  - **啟用中仍常為 0**：啟用中的站點小時裡，20.41% 完全沒有借還（多在深夜）。
  - **還入少 75 筆**：還車時間超出 2025-12-31 23 時的 75 筆旅次（2026-01-01 00–07 時的跨年還車，以及 1 筆 2034-08-26 的錯誤紀錄）放不進時間格，沒有計入。
  - **無效借還（d）只在 2025-08 以後**：資料來源在 2025-08 以前就沒有 5 分鐘內的同站還車紀錄，`rent_invalid`／`return_invalid` 不能做跨期比較。
  - **臺大人文館**的 `station_id` 暫用 `-2`，等 `stations_in_field` 補上正確 ID 後需從 v6 起重跑。
  - 研究範圍以 2024 年站位快照建立；2025 年才有的其他新站（如 `羅斯福路三段283巷`）不在範圍內，和它的往來算在範圍外。
- Definition for each variable:
  - `station_id`：StationID（9 碼；臺大人文館為 `-2`）。
  - `station_name`：站名，含 `YouBike2.0_` 前綴。
  - `datetime`：該小時的開始時間 `YYYY-MM-DD HH:00:00`；例如 08:00:00 代表 08:00–08:59。
  - `in_service`：該站在該小時是否已啟用（1／0）。
  - `rent_total`：該站該小時的借出筆數（以借車時間計）。
    - `rent_to_outside`：借出後還到範圍外（trip_group b，台大借外還）。
    - `rent_to_field`：借出後還到範圍內（c，台大借台大還）。
    - `rent_invalid`：無效借還（d，同站借還且時長 ≤ 3 分鐘）。
  - `return_total`：該站該小時的還入筆數（以還車時間計）。
    - `return_from_outside`：從範圍外借來（a，外借台大還）。
    - `return_from_field`：從範圍內借來（c）。
    - `return_invalid`：無效借還（d）。
  - `net_flow`：`return_total` − `rent_total`，正值 = 該小時騎乘使車輛增加。
- Data path:
  - `data/processed/youbike_station_hourly_flow/youbike_station_hourly_flow_2024_2025.csv`（1,894,752 列 × 13 欄，約 165 MB，列在本資料夾 `.gitignore`，需透過 repo 以外的管道分享）
  - `data/processed/youbike_station_hourly_flow/youbike_station_hourly_flow_example.csv`（範例：捷運公館站(2號出口) 從 2025-09-01 00 時起連續 100 小時；可以 commit）
- Source data: `data/processed/youbike_traffic/YouBike_traffic_2024_v6.csv`、`YouBike_traffic_2025_v6.csv`（說明見 [YouBike_traffic_doc.ipynb](../youbike_traffic/YouBike_traffic_doc.ipynb)），以及 `data/processed/stations_in_field.csv`。
- Processing code: [data/code/youbike_station_hourly_flow.ipynb](../../code/youbike_station_hourly_flow.ipynb)
- Role: 主檔 = `analysis`；範例檔 = 檢視格式用，不作分析樣本。
- Verification status: 2026-11-08 在全新的 in-process IPython shell 由上而下執行本 notebook（本機沒有 Jupyter）：SHA-256、欄位、列數、站點 × 小時唯一且完整、時間範圍、計數非負、各類加總與淨流量、未啟用時為 0、各欄加總等於產生程式的紀錄、範例檔內容，全部通過（見 Assertions）。
- SHA-256:
  - 主檔：`98a965038ff25c14764bfc8ceb8e84ca098cc37119ff37f46d276a621cd583a9`
  - 範例檔：`a680a38429fde8bd1f033ac831870f3b52361725e0bd44da45ac8485fa84a0be`
- Downstream analysis input: `data/processed/youbike_station_hourly_flow/youbike_station_hourly_flow_2024_2025.csv`（與 Data path 相同）


In [1]:
from pathlib import Path
import hashlib
import pandas as pd


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "AGENTS.md").is_file() and (candidate / "templates").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook within the student project containing AGENTS.md and templates/.")


PROJECT_ROOT = find_project_root()


def project_file(relative_path, area, must_exist=True):
    # Explicit project-relative paths only; never select the first matching file.
    rel = Path(relative_path)
    if rel.is_absolute() or ".." in rel.parts or "<" in str(rel):
        raise ValueError(f"Replace the placeholder with an exact project-relative path: {relative_path}")
    path = (PROJECT_ROOT / rel).resolve()
    allowed = (PROJECT_ROOT / "data" / area).resolve()
    if allowed == PROJECT_ROOT or not allowed.is_relative_to(PROJECT_ROOT):
        raise ValueError(f"data/{area}/ resolves outside the project root.")
    if not path.is_relative_to(allowed):
        raise ValueError(f"Expected a file inside data/{area}/: {relative_path}")
    if must_exist and not path.is_file():
        raise FileNotFoundError(path)
    return path


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def verify_input_sha256(path, expected):
    # Bind this run to the exact file inspected when the notebook was prepared.
    if (not isinstance(expected, str) or len(expected) != 64
            or any(char not in "0123456789abcdefABCDEF" for char in expected)):
        raise ValueError("Fill the expected SHA-256 from the inspected input before loading data.")
    actual = sha256_file(path)
    if actual != expected.lower():
        raise ValueError(f"Input changed: identify the new version before loading {path.name}.")
    return actual


import numpy as np
from IPython.display import display
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## Change for Your Own Data

下方設定綁定本文件描述的主檔與範例檔；路徑不存在或 SHA-256 不符就停止，不會改讀其他版本。`EXPECTED_TOTALS` 是產生程式記錄的各欄加總，用來核對檔案內容。

In [2]:
# Exact files documented here (produced by data/code/youbike_station_hourly_flow.ipynb).
DIR = "data/processed/youbike_station_hourly_flow"
MAIN = {"path": f"{DIR}/youbike_station_hourly_flow_2024_2025.csv", "rows": 1_894_752,
        "sha256": "98a965038ff25c14764bfc8ceb8e84ca098cc37119ff37f46d276a621cd583a9"}
EXAMPLE = {"path": f"{DIR}/youbike_station_hourly_flow_example.csv", "rows": 100,
           "sha256": "a680a38429fde8bd1f033ac831870f3b52361725e0bd44da45ac8485fa84a0be",
           "station_id": "500101022", "start": "2025-09-01 00:00:00"}
EXPECTED_COLUMNS = ["station_id", "station_name", "datetime", "in_service",
                    "rent_total", "rent_to_outside", "rent_to_field", "rent_invalid",
                    "return_total", "return_from_outside", "return_from_field", "return_invalid", "net_flow"]
COUNT_COLUMNS = EXPECTED_COLUMNS[4:]
RENT_PARTS = ["rent_to_outside", "rent_to_field", "rent_invalid"]
RETURN_PARTS = ["return_from_outside", "return_from_field", "return_invalid"]
N_STATIONS, N_HOURS = 108, 17_544
# Column totals recorded by the processing notebook (trip counts from the v6 files).
EXPECTED_TOTALS = {"rent_total": 14_008_037, "rent_to_outside": 4_224_078, "rent_to_field": 9_781_255, "rent_invalid": 2_704,
                   "return_total": 13_672_230, "return_from_outside": 3_888_312, "return_from_field": 9_781_214,
                   "return_invalid": 2_704, "net_flow": -335_807}
READ = {"dtype": {"station_id": str, "station_name": str, "datetime": str}, "encoding": "utf-8-sig"}

## Summary

先核對 SHA-256 再讀入。`_dt` 是為了核對而暫時解析的時間欄，不寫回檔案。

In [3]:
path = project_file(MAIN["path"], "processed")
MAIN["fingerprint"] = verify_input_sha256(path, MAIN["sha256"])
df = pd.read_csv(path, **READ)
print({"data_path": MAIN["path"], "rows": len(df), "schema": {c: str(t) for c, t in df.dtypes.items()},
       "sha256": MAIN["fingerprint"]})
example_path = project_file(EXAMPLE["path"], "processed")
EXAMPLE["fingerprint"] = verify_input_sha256(example_path, EXAMPLE["sha256"])
example = pd.read_csv(example_path, **READ)
print({"data_path": EXAMPLE["path"], "rows": len(example), "sha256": EXAMPLE["fingerprint"]})
df["_dt"] = pd.to_datetime(df["datetime"], format="%Y-%m-%d %H:%M:%S")   # for checks only

{'data_path': 'data/processed/youbike_station_hourly_flow/youbike_station_hourly_flow_2024_2025.csv', 'rows': 1894752, 'schema': {'station_id': 'object', 'station_name': 'object', 'datetime': 'object', 'in_service': 'int64', 'rent_total': 'int64', 'rent_to_outside': 'int64', 'rent_to_field': 'int64', 'rent_invalid': 'int64', 'return_total': 'int64', 'return_from_outside': 'int64', 'return_from_field': 'int64', 'return_invalid': 'int64', 'net_flow': 'int64'}, 'sha256': '98a965038ff25c14764bfc8ceb8e84ca098cc37119ff37f46d276a621cd583a9'}
{'data_path': 'data/processed/youbike_station_hourly_flow/youbike_station_hourly_flow_example.csv', 'rows': 100, 'sha256': 'a680a38429fde8bd1f033ac831870f3b52361725e0bd44da45ac8485fa84a0be'}


### Sample

In [4]:
print("主檔依檔案順序的前 10 列（臺大人文館 2024-01-01，尚未啟用，所以全為 0）")
display(df[EXPECTED_COLUMNS].head(10))
print("範例檔的第 7–20 列（捷運公館站(2號出口)，2025-09-01 06–19 時）")
display(example.iloc[6:20])

主檔依檔案順序的前 10 列（臺大人文館 2024-01-01，尚未啟用，所以全為 0）


,station_id,station_name,datetime,in_service,rent_total,rent_to_outside,rent_to_field,rent_invalid,return_total,return_from_outside,return_from_field,return_invalid,net_flow
0,-2,YouBike2.0_臺大人文館,2024-01-01 00:00:00,0,0,0,0,0,0,0,0,0,0
1,-2,YouBike2.0_臺大人文館,2024-01-01 01:00:00,0,0,0,0,0,0,0,0,0,0
2,-2,YouBike2.0_臺大人文館,2024-01-01 02:00:00,0,0,0,0,0,0,0,0,0,0
3,-2,YouBike2.0_臺大人文館,2024-01-01 03:00:00,0,0,0,0,0,0,0,0,0,0
4,-2,YouBike2.0_臺大人文館,2024-01-01 04:00:00,0,0,0,0,0,0,0,0,0,0
5,-2,YouBike2.0_臺大人文館,2024-01-01 05:00:00,0,0,0,0,0,0,0,0,0,0
6,-2,YouBike2.0_臺大人文館,2024-01-01 06:00:00,0,0,0,0,0,0,0,0,0,0
7,-2,YouBike2.0_臺大人文館,2024-01-01 07:00:00,0,0,0,0,0,0,0,0,0,0
8,-2,YouBike2.0_臺大人文館,2024-01-01 08:00:00,0,0,0,0,0,0,0,0,0,0
9,-2,YouBike2.0_臺大人文館,2024-01-01 09:00:00,0,0,0,0,0,0,0,0,0,0


範例檔的第 7–20 列（捷運公館站(2號出口)，2025-09-01 06–19 時）


,station_id,station_name,datetime,in_service,rent_total,rent_to_outside,rent_to_field,rent_invalid,return_total,return_from_outside,return_from_field,return_invalid,net_flow
6,500101022,YouBike2.0_捷運公館站(2號出口),2025-09-01 06:00:00,1,49,14,35,0,20,13,7,0,-29
7,500101022,YouBike2.0_捷運公館站(2號出口),2025-09-01 07:00:00,1,69,30,39,0,49,19,30,0,-20
8,500101022,YouBike2.0_捷運公館站(2號出口),2025-09-01 08:00:00,1,60,15,45,0,57,26,31,0,-3
9,500101022,YouBike2.0_捷運公館站(2號出口),2025-09-01 09:00:00,1,39,16,23,0,39,21,18,0,0
10,500101022,YouBike2.0_捷運公館站(2號出口),2025-09-01 10:00:00,1,73,26,47,0,73,32,41,0,0
11,500101022,YouBike2.0_捷運公館站(2號出口),2025-09-01 11:00:00,1,120,42,78,0,128,55,73,0,8
12,500101022,YouBike2.0_捷運公館站(2號出口),2025-09-01 12:00:00,1,197,81,116,0,191,75,116,0,-6
13,500101022,YouBike2.0_捷運公館站(2號出口),2025-09-01 13:00:00,1,86,23,63,0,86,36,50,0,0
14,500101022,YouBike2.0_捷運公館站(2號出口),2025-09-01 14:00:00,1,97,25,72,0,133,57,76,0,36
15,500101022,YouBike2.0_捷運公館站(2號出口),2025-09-01 15:00:00,1,112,46,66,0,148,45,103,0,36


### Summary Stats

In [5]:
def fmt_int(x): return f"{int(round(x)):,}"
def fmt2(x): return f"{x:,.2f}"


def numeric_summary(frame):
    return pd.DataFrame({c: {"N": fmt_int(frame[c].notna().sum()), "缺值 N": fmt_int(frame[c].isna().sum()),
                             "平均數": fmt2(frame[c].mean()), "標準差": fmt2(frame[c].std()), "最小值": fmt_int(frame[c].min()),
                             "P25": fmt2(frame[c].quantile(.25)), "中位數": fmt2(frame[c].median()),
                             "P75": fmt2(frame[c].quantile(.75)), "最大值": fmt_int(frame[c].max()),
                             "合計": fmt_int(frame[c].sum())} for c in COUNT_COLUMNS}).T


print(f"表 1　每站每小時流量（全部 {len(df):,} 列）")
display(numeric_summary(df))
active = df[df["in_service"] == 1]
print(f"表 2　每站每小時流量（只含 in_service = 1 的 {len(active):,} 列）")
display(numeric_summary(active))

print("表 3　類別欄位")
display(pd.DataFrame({
    "station_id": {"N": fmt_int(df["station_id"].notna().sum()), "不同值數": df["station_id"].nunique(),
                   "每站列數": f"{df.groupby('station_id').size().min():,}～{df.groupby('station_id').size().max():,}"},
    "datetime": {"N": fmt_int(df["datetime"].notna().sum()), "不同值數": f"{df['datetime'].nunique():,}",
                 "每小時列數": f"{df.groupby('datetime').size().min()}～{df.groupby('datetime').size().max()}"},
    "in_service": {"N": fmt_int(len(df)), "不同值數": df["in_service"].nunique(),
                   "每小時列數": f"1：{(df['in_service'] == 1).sum():,}（{(df['in_service'] == 1).mean():.2%}）；0：{(df['in_service'] == 0).sum():,}"},
}).T)

zero_share = ((active["rent_total"] == 0) & (active["return_total"] == 0)).mean()
print(f"啟用中的站點小時裡，完全沒有借還的占 {zero_share:.2%}。")
by_station = df.groupby(["station_id", "station_name"])[["rent_total", "return_total", "net_flow"]].sum()
by_station["hours_in_service"] = df.groupby(["station_id", "station_name"])["in_service"].sum()
print("表 4　借出量前 10 站（兩年合計）")
display(by_station.sort_values("rent_total", ascending=False).head(10).map(fmt_int))
first_active = df[df["in_service"] == 1].groupby(["station_id", "station_name"])["datetime"].min()
print("表 5　啟用時間晚於 2024-01-01 00 時的站")
display(first_active[first_active > "2024-01-01 00:00:00"].rename("in_service 起始").to_frame())

表 1　每站每小時流量（全部 1,894,752 列）


,N,缺值 N,平均數,標準差,最小值,P25,中位數,P75,最大值,合計
rent_total,"1,894,752",0,7.39,13.91,0,0.00,3.00,9.00,283,"14,008,037"
rent_to_outside,"1,894,752",0,2.23,5.18,0,0.00,1.00,2.00,147,"4,224,078"
rent_to_field,"1,894,752",0,5.16,10.18,0,0.00,2.00,6.00,271,"9,781,255"
rent_invalid,"1,894,752",0,0.00,0.04,0,0.00,0.00,0.00,5,"2,704"
return_total,"1,894,752",0,7.22,14.78,0,0.00,2.00,8.00,380,"13,672,230"
return_from_outside,"1,894,752",0,2.05,4.94,0,0.00,0.00,2.00,143,"3,888,312"
return_from_field,"1,894,752",0,5.16,11.07,0,0.00,2.00,6.00,323,"9,781,214"
return_invalid,"1,894,752",0,0.00,0.04,0,0.00,0.00,0.00,5,"2,704"
net_flow,"1,894,752",0,-0.18,8.02,-204,-1.00,0.00,1.00,249,"-335,807"


表 2　每站每小時流量（只含 in_service = 1 的 1,834,704 列）


,N,缺值 N,平均數,標準差,最小值,P25,中位數,P75,最大值,合計
rent_total,"1,834,704",0,7.64,14.08,0,0.00,3.00,9.00,283,"14,008,037"
rent_to_outside,"1,834,704",0,2.30,5.25,0,0.00,1.00,2.00,147,"4,224,078"
rent_to_field,"1,834,704",0,5.33,10.30,0,0.00,2.00,6.00,271,"9,781,255"
rent_invalid,"1,834,704",0,0.00,0.04,0,0.00,0.00,0.00,5,"2,704"
return_total,"1,834,704",0,7.45,14.96,0,0.00,3.00,9.00,380,"13,672,230"
return_from_outside,"1,834,704",0,2.12,5.00,0,0.00,1.00,2.00,143,"3,888,312"
return_from_field,"1,834,704",0,5.33,11.21,0,0.00,2.00,6.00,323,"9,781,214"
return_invalid,"1,834,704",0,0.00,0.04,0,0.00,0.00,0.00,5,"2,704"
net_flow,"1,834,704",0,-0.18,8.15,-204,-1.00,0.00,1.00,249,"-335,807"


表 3　類別欄位


,N,不同值數,每站列數,每小時列數
station_id,"1,894,752",108,"17,544～17,544",NaN
datetime,"1,894,752","17,544",NaN,108～108
in_service,"1,894,752",2,NaN,"1：1,834,704（96.83%）；0：60,048"


啟用中的站點小時裡，完全沒有借還的占 20.41%。
表 4　借出量前 10 站（兩年合計）


,,rent_total,return_total,net_flow,hours_in_service
station_id,station_name,,,,
500101022,YouBike2.0_捷運公館站(2號出口),"1,059,785","1,125,624","65,839","17,544"
500101181,YouBike2.0_捷運公館站(3號出口),"661,328","700,731","39,403","17,544"
500101001,YouBike2.0_捷運科技大樓站,"650,484","629,552","-20,932","17,544"
500119075,YouBike2.0_臺大第一活動中心西南側,"396,275","426,877","30,602","17,544"
500119090,YouBike2.0_臺大新體育館東南側,"307,849","314,664","6,815","17,544"
500119050,YouBike2.0_臺大小福樓東側,"297,480","316,283","18,803","17,544"
500101032,YouBike2.0_臺大男一舍前,"283,023","272,953","-10,070","17,544"
500119067,YouBike2.0_臺大社科院西側,"252,191","251,646",-545,"17,544"
500119049,YouBike2.0_臺大女九舍西南側,"236,957","246,904","9,947","17,544"


表 5　啟用時間晚於 2024-01-01 00 時的站


,,in_service 起始
station_id,station_name,
-2,YouBike2.0_臺大人文館,2025-02-04 00:00:00
500101166,YouBike2.0_捷運科技大樓站(台北教育大學),2024-01-02 00:00:00
500101234,YouBike2.0_和平東路二段90巷,2024-03-19 00:00:00
500101240,YouBike2.0_和平東路二段76巷,2024-09-20 00:00:00
500101250,YouBike2.0_國立臺北教育大學創意館,2024-12-13 00:00:00
500106135,YouBike2.0_羅斯福路四段108巷口,2024-10-15 00:00:00
500119059,YouBike2.0_臺大共同教室北側,2024-01-02 00:00:00
500119092,YouBike2.0_臺大管理學院一館,2024-08-13 00:00:00
500119093,YouBike2.0_臺大禮賢樓西側,2024-08-16 00:00:00


### Assertions

檢查由產生規則支持的性質：欄位、列數、站點 × 小時唯一且完整、時間範圍與整點、站名一致、計數非負、各類加總等於總量、淨流量定義、未啟用時借還為 0 且啟用後不再變回 0、欄位加總等於產生程式的紀錄、無效借還只出現在 2025-08 以後，以及範例檔等於主檔對應的 100 列。

In [6]:
assert list(df.columns[:13]) == EXPECTED_COLUMNS, "schema differs"
assert len(df) == MAIN["rows"] == N_STATIONS * N_HOURS
assert not df.duplicated(["station_id", "datetime"]).any(), "station × hour is not unique"
assert df["station_id"].nunique() == N_STATIONS and df["datetime"].nunique() == N_HOURS
assert df["_dt"].min() == pd.Timestamp("2024-01-01 00:00") and df["_dt"].max() == pd.Timestamp("2025-12-31 23:00")
assert (df["_dt"].dt.minute == 0).all() and (df["_dt"].dt.second == 0).all()
assert df.groupby("station_id")["station_name"].nunique().max() == 1
assert df["station_id"].str.fullmatch(r"\d{9}|-2").all()
assert df["in_service"].isin([0, 1]).all()
assert df[COUNT_COLUMNS].notna().all().all()
assert (df[COUNT_COLUMNS[:-1]] >= 0).all().all(), "negative count"
assert (df["rent_total"] == df[RENT_PARTS].sum(axis=1)).all()
assert (df["return_total"] == df[RETURN_PARTS].sum(axis=1)).all()
assert (df["net_flow"] == df["return_total"] - df["rent_total"]).all()
assert (df.loc[df["in_service"] == 0, ["rent_total", "return_total"]] == 0).all().all(), "flow before in_service"
# Once a station is in service it stays in service.
ordered = df.sort_values(["station_id", "_dt"])
assert (ordered.groupby("station_id")["in_service"].diff().fillna(0) >= 0).all(), "in_service switches back to 0"
assert df[COUNT_COLUMNS].sum().to_dict() == EXPECTED_TOTALS, "totals differ from the processing record"
assert (df.loc[df["rent_invalid"] + df["return_invalid"] > 0, "datetime"] >= "2025-08-01").all(), "invalid flow before 2025-08"
expected_example = (df[(df["station_id"] == EXAMPLE["station_id"]) & (df["datetime"] >= EXAMPLE["start"])]
                    [EXPECTED_COLUMNS].head(EXAMPLE["rows"]).reset_index(drop=True))
pd.testing.assert_frame_equal(example, expected_example)
assert sha256_file(path) == MAIN["fingerprint"] and sha256_file(example_path) == EXAMPLE["fingerprint"]
print({"data_path": MAIN["path"], "rows": len(df), "columns": 13, "sha256": MAIN["fingerprint"], "status": "configured checks passed"})
print({"data_path": EXAMPLE["path"], "rows": len(example), "columns": 13, "sha256": EXAMPLE["fingerprint"],
       "status": "equals the matching 100 rows of the main file"})

{'data_path': 'data/processed/youbike_station_hourly_flow/youbike_station_hourly_flow_2024_2025.csv', 'rows': 1894752, 'columns': 13, 'sha256': '98a965038ff25c14764bfc8ceb8e84ca098cc37119ff37f46d276a621cd583a9', 'status': 'configured checks passed'}
{'data_path': 'data/processed/youbike_station_hourly_flow/youbike_station_hourly_flow_example.csv', 'rows': 100, 'columns': 13, 'sha256': 'a680a38429fde8bd1f033ac831870f3b52361725e0bd44da45ac8485fa84a0be', 'status': 'equals the matching 100 rows of the main file'}


### 本次文件核對的結論與限制

- 主檔與範例檔都與產生程式的紀錄一致，所有 assertion 通過。
- 每站每小時的借出量平均 7.39 筆（中位數 3），還入量平均 7.22 筆；單一站點單一小時最多借出 283 筆、還入 380 筆，淨流量介於 −204 到 +249。分佈高度右偏，大量站點小時接近 0。
- 兩年合計：借出 14,008,037 筆（台大借外還 4,224,078、台大借台大還 9,781,255、無效 2,704），還入 13,672,230 筆（外借台大還 3,888,312、台大借台大還 9,781,214、無效 2,704），淨流量 −335,807。
- 借出量最多的是捷運公館站(2號出口)（1,059,785 筆，淨流量 +65,839）、公館站(3號出口) 和捷運科技大樓站。
- 本文件只描述與核對，沒有修改任何資料。
